# CARE autoencoder v2 training on Colab (all farms)

Trains the three cleaned-input candidates of `docs/care-autoencoder-v2-protocol.md`
(`clean`, `clean_long`, `clean_long_delta`) on all 95 CARE datasets. It only trains and
saves scores; candidates are compared locally by leave-one-farm-out CARE score.

1. **Runtime → Change runtime type → T4 GPU.** The longer-training candidates make a GPU
   worthwhile here (CPU would take several hours).
2. **Runtime → Run all.** Upload the input ZIP from `scripts/package_care_colab.py` when asked.
3. Allow Google Drive access when asked. Results go to
   `MyDrive/HydroPredict/care-autoencoder-v2`; rerunning resumes from the next unfinished
   dataset, so a disconnect loses at most one dataset.

Expect roughly 1-2 hours on a T4: each CSV is read once and all three candidates are
trained from it. The CARE data (5.5 GB) comes from your Drive copy if the v1 run saved
one, otherwise from Zenodo, and is checked against its published MD5 either way.

In [ ]:
import hashlib
import json
import subprocess
import sys
import tempfile
import urllib.request
import zipfile
from pathlib import Path

from google.colab import files
from tqdm.auto import tqdm

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly the CARE Colab input ZIP.")
WORK = Path(tempfile.mkdtemp(prefix="care-"))
INPUT = WORK / "inputs"


def extract_owned_zip(data, destination):
    destination.mkdir(parents=True, exist_ok=True)
    path = destination.with_suffix(".zip")
    path.write_bytes(data)
    with zipfile.ZipFile(path) as bundle:
        for entry in bundle.infolist():
            target = (destination / entry.filename).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError("Unsafe archive path")
        bundle.extractall(destination)


extract_owned_zip(next(iter(uploaded.values())), INPUT)
manifest = json.loads((INPUT / "input_manifest.json").read_text())
for name, expected in manifest["files_sha256"].items():
    if hashlib.sha256((INPUT / name).read_bytes()).hexdigest() != expected:
        raise ValueError(f"Input file changed: {name}")
PROJECT = WORK / "project"
extract_owned_zip((INPUT / "project.zip").read_bytes(), PROJECT)
print("Source commit:", manifest["source_commit"])


def run_streamed(command, total=0, done=0, desc=None):
    # Show the command's lines and a per-dataset progress bar.
    bar = tqdm(total=total, initial=done, desc=desc, unit="dataset") if total else None
    with subprocess.Popen(
        command, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    ) as process:
        for line in process.stdout:
            if "Warning" in line:
                continue
            (bar.write if bar else print)(line.rstrip())
            if bar and " epochs, " in line:
                bar.update(1)
    if bar:
        bar.close()
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, command)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT)], check=True)

In [ ]:
import torch

GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("PyTorch:", torch.__version__, "| GPU:", GPU or "none (CPU run)")
if GPU is None:
    print("No GPU: Runtime → Change runtime type → T4 GPU, then Runtime → Run all.")

## Get and verify CARE

Colab's own disk is wiped when the runtime ends, so the dataset is cached in Drive.
Set `CACHE_IN_DRIVE = False` below to skip saving that 5.5 GB copy. All three farms
are extracted (about 20 GB on Colab's disk).

In [ ]:
import shutil

from google.colab import drive

drive.mount("/content/drive")
CACHE_IN_DRIVE = True
CACHE = Path("/content/drive/MyDrive/HydroPredict/CARE_To_Compare.zip")
CACHE.parent.mkdir(parents=True, exist_ok=True)
DATA_ZIP = Path("/content/CARE_To_Compare.zip")
TOTAL = manifest["zenodo_bytes"]


def copy_with_progress(source, target, desc):
    with (
        source.open("rb") as src,
        target.open("wb") as dst,
        tqdm(total=TOTAL, unit="B", unit_scale=True, desc=desc) as bar,
    ):
        for block in iter(lambda: src.read(16 * 1024 * 1024), b""):
            dst.write(block)
            bar.update(len(block))


cached = CACHE.exists() and CACHE.stat().st_size == TOTAL
if cached and not (DATA_ZIP.exists() and DATA_ZIP.stat().st_size == TOTAL):
    copy_with_progress(CACHE, DATA_ZIP, "Copy from Drive")
done = DATA_ZIP.stat().st_size if DATA_ZIP.exists() else 0
if done < TOTAL:
    # Resume a partial download when the server honours the byte range.
    headers = {"Range": f"bytes={done}-"} if done else {}
    request = urllib.request.Request(manifest["zenodo_url"], headers=headers)
    with urllib.request.urlopen(request) as response:
        resumed = response.status == 206
        with (
            DATA_ZIP.open("ab" if resumed else "wb") as out,
            tqdm(
                total=TOTAL,
                initial=done if resumed else 0,
                unit="B",
                unit_scale=True,
                desc="Download",
            ) as bar,
        ):
            for block in iter(lambda: response.read(1024 * 1024), b""):
                out.write(block)
                bar.update(len(block))
md5 = hashlib.md5()
with (
    DATA_ZIP.open("rb") as stream,
    tqdm(total=TOTAL, unit="B", unit_scale=True, desc="Verify") as bar,
):
    for block in iter(lambda: stream.read(16 * 1024 * 1024), b""):
        md5.update(block)
        bar.update(len(block))
if md5.hexdigest() != manifest["zenodo_md5"]:
    DATA_ZIP.unlink()
    hint = f" Delete the Drive copy {CACHE} and rerun." if cached else " Rerun this cell."
    raise ValueError("CARE archive does not match the published MD5." + hint)
if CACHE_IN_DRIVE and not cached:
    free = shutil.disk_usage(CACHE.parent).free
    if free > TOTAL + 1024**3:
        copy_with_progress(DATA_ZIP, CACHE.with_suffix(".partial"), "Save to Drive")
        CACHE.with_suffix(".partial").rename(CACHE)
    else:
        print(f"Not caching in Drive: only {free / 1024**3:.1f} GB free.")
DATA = Path("/content/care")
farms = [manifest["parity"]["farm"], *manifest["train_farms"]]
with zipfile.ZipFile(DATA_ZIP) as bundle:
    members = [m for m in bundle.infolist() if any(f"/{farm}/" in m.filename for farm in farms)]
    with tqdm(
        total=sum(m.file_size for m in members), unit="B", unit_scale=True, desc="Extract"
    ) as bar:
        for member in members:
            bundle.extract(member, DATA)
            bar.update(member.file_size)
ROOT = DATA / "CARE_To_Compare"
print("Verified and extracted:", farms)

## Device parity check

Trains the `clean` candidate on one Farm A dataset here and compares it with the same
training on the laptop. GPU arithmetic differs slightly from CPU, so small differences
are expected; a weak correlation means this runtime doesn't reproduce the local code,
and the run stops before spending GPU time.

In [ ]:
import numpy as np

RUN_ROOT = Path("/content/drive/MyDrive/HydroPredict/care-autoencoder-v2")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
SCRIPT = [sys.executable, "-m", "scripts.train_care_autoencoder_v2", str(ROOT)]
parity = manifest["parity"]
PARITY_DIR = WORK / "parity"
run_streamed(
    SCRIPT
    + ["--farms", parity["farm"], "--events", str(parity["event_id"])]
    + ["--candidates", "clean", "--output", str(PARITY_DIR), "--device", "auto"],
    total=1,
    desc="Parity",
)
name = f"{parity['farm'].replace(' ', '_')}_{parity['event_id']}.npz"
ours = np.load(PARITY_DIR / "clean" / name)["scores"]
local = np.load(INPUT / "parity_reference.npz")["scores"]
both = ~np.isnan(ours) & ~np.isnan(local)
result = dict(
    gpu=GPU,
    unscored_match=bool((np.isnan(ours) == np.isnan(local)).all()),
    correlation=float(np.corrcoef(ours[both], local[both])[0, 1]),
    median_relative_difference=float(np.median(np.abs(ours[both] - local[both]) / local[both])),
)
(RUN_ROOT / "parity.json").write_text(json.dumps(result, indent=2))
print(result)
if not result["unscored_match"] or result["correlation"] < 0.95:
    raise RuntimeError("This runtime does not reproduce the local model closely enough.")

## Train all candidates on all farms

95 datasets, three candidates each. Progress is saved to Drive after every dataset; if
Colab disconnects, reconnect and **Run all** again: finished datasets are skipped.

In [ ]:
CANDIDATES = ["clean", "clean_long", "clean_long_delta"]
farms = manifest["train_farms"]
names = [
    f"{farm.replace(' ', '_')}_{path.stem}"
    for farm in farms
    for path in (ROOT / farm / "datasets").glob("*.csv")
]
finished = sum(all((RUN_ROOT / c / f"{n}.npz").exists() for c in CANDIDATES) for n in names)
run_streamed(
    SCRIPT + ["--farms", *farms, "--output", str(RUN_ROOT), "--device", "auto"],
    total=len(names),
    done=finished,
    desc="Train",
)
(RUN_ROOT / "source_commit.txt").write_text(manifest["source_commit"])

## Training summary

`epochs` shows whether the longer candidates used their extra room (v1 stopped at the
40-epoch cap on 29 of 37 Farm A/B datasets). `dropped` is the number of quasi-constant
channels removed per dataset.

In [ ]:
import pandas as pd

audit = pd.DataFrame(json.loads((RUN_ROOT / "training.json").read_text()))
cap = audit.candidate.map({"clean": 40, "clean_long": 200, "clean_long_delta": 200})
audit["hit_cap"] = audit.epochs >= cap
summary = audit.groupby(["candidate", "farm"]).agg(
    datasets=("event_id", "nunique"),
    median_epochs=("epochs", "median"),
    hit_cap=("hit_cap", "sum"),
    median_dropped=("dropped", "median"),
    minutes=("seconds", lambda s: round(s.sum() / 60, 1)),
)
print(
    "Datasets with all candidates:",
    sum(all((RUN_ROOT / c / f"{n}.npz").exists() for c in CANDIDATES) for n in names),
    "of",
    len(names),
)
summary

## Bring the results back

Download the ZIP below and tell Claude where it is saved. It holds every candidate's
score files and model weights, `training.json`, `config.json`, `parity.json` and the
source commit. Drive keeps the persistent copy.

In [ ]:
import shutil

export = shutil.make_archive("/content/care-autoencoder-v2", "zip", root_dir=RUN_ROOT)
files.download(export)